# 5 Exercises: the handed-out notebook, and a study of your own

Notebook 04 ran one search and read it for you. Here you drive.

## What is asked of you

1. **Part 1: run this notebook as it is.** A baseline, an encoder comparison, one search
   over a given space, its winner rerun, and a noise ablation. You change nothing; you
   read what comes out and answer the questions for yourself.
2. **Part 2: one study of your own.** One idea, a design that can test it, a search, and
   the result against a baseline. You do this in the folder [`exercises/`](exercises/),
   next to this notebook; Part 2 below walks you through it.
3. **Hand in your runs** before the next lesson, with one command: `make share LESSON=1`.

The next lesson opens on the runs of the whole class, yours among them. A study that
disappointed is a result worth bringing: "I expected X, measured Y, and here is the spread
between runs that makes me believe it" is the habit this lesson exists for.

## Where your runs go

Nothing here is automatic or hidden. There are three steps, and you start each of them:

| step | what does it | what happens |
|---|---|---|
| **store** | a line in this notebook: `RUNS.log(...)` for one run, `search(...)` for a search | the run is written as one row (a *trial*) in a file on your own machine, `artefacts/optuna/studies.db`. Nothing leaves your machine. |
| **look** | `make optuna-dashboard`, in a terminal | opens that file in your browser: every run, its params, its accuracy, its architecture |
| **hand in** | `make share LESSON=1`, in a terminal | reads the runs of lesson 1 from that file, writes each as a small JSON file, and sends them to the class repository on GitHub as a pull request from your account. No model weights and no data are sent. |

The rows are grouped in *studies*, and a study has a name. Part 1 writes to two:
`lesson1-notebook` (the trials of the search) and `lesson1-notebook-runs` (the baseline,
the encoder comparison, the winner, the noise ablation). Part 2 writes to
`lesson1-study-<your study>` and `lesson1-study-<your study>-runs`. `make share` hands in
exactly those four.

**Nothing in Part 1 is seeded.** Every model starts from other random weights, on your
machine and on everyone else's, so your numbers will not be your neighbour's. That is the
point: fifteen baselines side by side show how much one model varies from run to run, and
so how large a difference has to be before it means anything.

Everything starts on **DTD**: 47 texture classes, roughly twenty points of headroom, and
an encoder axis that actually moves.

## Setup: the caches, the helpers and the search

Three cells of setup. You do not have to read them to do the exercises; this is what they
give you:

| name | what it does |
|---|---|
| `loaders(dataset, encoder)` | the train and test loader of a cache |
| `run(model, train_dl, test_dl)` | trains a model for `EPOCHS` epochs and returns the fit: accuracy, seconds, loss curves |
| `run_spread(make_model, ...)` | the same, `REPEATS` times, each from other random weights |
| `compare({name: runs, ...})` | a picture and a table of repeated runs: mean, spread, and the difference with the first entry |
| `make_head(config, dim, n_classes)` | builds the head a config describes |
| `search(space, name)` | runs `N_TRIALS` trials of a search space as one Optuna study |
| `RUNS` | the study `lesson1-notebook-runs`; `RUNS.log(...)` stores one run in it |

In [ ]:
import sys
import time
from functools import cache

import datasets
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from loguru import logger
from notebooktester import param
from torch import nn
from torch.utils.data import DataLoader
from vectormesh import VectorCache
from vectormesh.components import (
    Gate,
    GaussianNoise,
    NeuralNet,
    Norm,
    Projection,
    Serial,
    Skip,
)
from vectormesh.data import Collate, OneHot

from plotting import (
    plot_learning_curves,
    result_bars,
    search_parallel_plot,
    study_loop,
)

from hpo.measure import Architecture, Comparison, Fit, Repeats, Scorer, Training, repeat
from hpo.record import Idea, RunRecord, Store, StudyLog
from hpo.search import Budget, Choice, Float, Int, Space

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()
# a search shows one progress bar; Optuna's line per trial stays off
optuna.logging.set_verbosity(optuna.logging.WARNING)

EPOCHS = param(15, test=1)
N_TRIALS = param(24, test=2)
REPEATS = param(3, test=2)

DATASET = "dtd"
ENCODERS = ["dinov2-small", "resnet-18", "mobilenet_v2_1.0_224"]

# Every published cache, by dataset and encoder. A study on another dataset or encoder
# picks its cache from here.
CACHES = {
    "dtd": {
        enc: f"pttrn-io/dtd-{enc}"
        for enc in [
            "dinov2-small",
            "dinov2-base",
            "dinov2-large",
            "resnet-18",
            "mobilenet_v2_1.0_224",
        ]
    },
    "eurosat": {enc: f"pttrn-io/eurosat-{enc}" for enc in ENCODERS},
    "oxford-flowers": {enc: f"pttrn-io/oxford-flowers-{enc}" for enc in ENCODERS},
    "fashion-mnist": {
        enc: f"pttrn-io/fashion-mnist-{enc}"
        for enc in ["dinov2-small", "mobilenet_v2_1.0_224"]
    },
}

# The values a string param can take, for the runs that are given their params (see
# hpo.record.runs.StudyLog).
CHOICES = {
    "dataset": list(CACHES),
    "encoder": sorted({enc for encoders in CACHES.values() for enc in encoders}),
    "activation": ["relu", "gelu", "tanh", "sigmoid", "none"],
    "norm": ["none", "batch", "layer"],
}

# The store is the file the runs are written to: artefacts/optuna/studies.db.
STORE = Store.default()
# Every study says which idea it investigates. Part 1 is the same for everyone, so its
# idea is the course's: "lesson 1 as handed out".
HANDED_OUT = Idea.handed_out(lesson=1)
# The study that holds the baseline, the encoder comparison, the winner and the noise
# ablation. RUNS.log(...) writes one run to it.
RUNS = StudyLog.runs(name="lesson1-notebook-runs", idea=HANDED_OUT, choices=CHOICES)

In [ ]:
def load(dataset: str, encoder: str, split: str) -> VectorCache:
    if encoder not in CACHES.get(dataset, {}):
        raise KeyError(
            f"no published cache for {dataset!r} with {encoder!r}; "
            f"CACHES has { ({d: list(e) for d, e in CACHES.items()}) }"
        )
    return VectorCache.from_hub(CACHES[dataset][encoder], split=split)


@cache
def shape_of(dataset: str, encoder: str) -> tuple[int, int]:
    """(vector width, number of classes) of a cache."""
    train = load(dataset, encoder, "train")
    return train.dim, train.num_classes


def loaders(
    dataset: str, encoder: str, batch_size: int = 64, n_train: int | None = None
):
    # n_train shrinks the training set for the data-size questions below.
    # The shuffle is NOT optional -- see the cell after this one.
    out = []
    for split in ("train", "test"):
        cache = load(dataset, encoder, split)
        ds = cache.dataset
        if n_train is not None and split == "train":
            ds = ds.shuffle(seed=0).select(range(n_train))
        oh = OneHot(
            num_classes=cache.num_classes, label_col="label", target_col="onehot"
        )
        col = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)
        out.append(
            DataLoader(
                ds.map(oh),
                batch_size=batch_size,
                shuffle=(split == "train"),
                collate_fn=col,
            )
        )
    return out


def classes_of(loader) -> list[str]:
    """The names of the classes of a loader's cache, in the order of their labels."""
    return list(loader.dataset.features["label"].names)


def run(
    model,
    train_dl,
    test_dl,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> Fit:
    """Train `model` and return the fit: the confusion on the test set after the last
    epoch, the seconds spent in training steps, and the curves. `fit.scores()` holds
    every metric: accuracy, macro recall, and recall, precision and F1 per class
    (`hpo.measure.metrics`); every run logs them all."""
    # hpo.measure.training.Training is vectormesh's Trainer, timed, and scored on the whole test
    # set after every epoch; progress=False inside it drops the tqdm bars, which is what
    # makes calling this once per run -- several times per exercise below -- cheap.
    training = Training(lr=lr, epochs=epochs, logdir=STORE.logdir())
    return training.fit(model, train_dl, test_dl, classes_of(test_dl))


# Every choice a head can make, with the value that leaves it out. A run logs all of them
# (see run_params), so each is an axis in the class's parallel-coordinates plot.
HEAD = {
    "n_hidden_layers": 0,
    "hidden_dim": 0,
    "activation": "gelu",
    "sigma": 0.0,
    "norm": "none",
    "dropout": 0.0,
    "skip": False,
    "gate": False,
    "lr": 1e-3,
}

ACTIVATIONS = {"gelu": nn.GELU, "relu": nn.ReLU, "tanh": nn.Tanh, "sigmoid": nn.Sigmoid}


def make_head(config: dict, dim: int, n_classes: int) -> Serial:
    # the same builder as notebook 04 §4.1: 0 hidden layers is the baseline's single Linear,
    # more layers with the defaults go through one NeuralNet call, which puts a GELU between
    # every Linear. activation (a key of ACTIVATIONS), norm ("batch", "layer" or "none"),
    # dropout (a probability), skip and gate change the hidden block; they do nothing
    # without hidden layers. lr goes to the optimizer, not here. Add an axis to your
    # param_space? Use it here, and give it a default in HEAD.
    c = {**HEAD, **config}
    n_hidden, width = c["n_hidden_layers"], c["hidden_dim"]
    if n_hidden == 0:
        head = nn.Linear(dim, n_classes)
    elif (
        c["activation"] == "gelu"
        and c["norm"] == "none"
        and c["dropout"] == 0
        and not c["skip"]
        and not c["gate"]
    ):
        head = NeuralNet([dim, *[width] * n_hidden], n_classes)
    else:
        layers, d = [], dim
        for _ in range(n_hidden):
            layers.append(nn.Linear(d, width))
            if c["norm"] != "none":
                layers.append(Norm(width, kind=c["norm"]))
            layers.append(ACTIVATIONS[c["activation"]]())
            if c["dropout"] > 0:
                layers.append(nn.Dropout(c["dropout"]))
            d = width
        block = Serial(layers)
        if c["skip"]:  # transform(norm(x)) + projection(norm(x)), see vectormesh's Skip
            block = Skip(block, in_size=dim, projection=Projection(dim, width))
        head = Serial(
            [block, *([Gate(width)] if c["gate"] else []), nn.Linear(width, n_classes)]
        )
    return Serial([GaussianNoise(sigma=c["sigma"]), head])


def run_params(dataset: str, encoder: str, config: dict | None = None, **extra) -> dict:
    """The params of a run: the vectors it starts from, every head choice, the budget.

    The head choices are the ones the model was built with: without hidden layers there
    is no hidden block, so hidden_dim, activation, norm, dropout, skip and gate are logged
    as off whatever the config drew."""
    head = {
        **HEAD,
        **{k: v for k, v in (config or {}).items() if k not in ("dataset", "encoder")},
    }
    if head["n_hidden_layers"] == 0:
        head.update(
            hidden_dim=0,
            activation="none",
            norm="none",
            dropout=0.0,
            skip=False,
            gate=False,
        )
    return {
        "dataset": dataset,
        "encoder": encoder,
        "embed_dim": shape_of(dataset, encoder)[0],
        "epochs": EPOCHS,
        **head,
        **extra,
    }


def run_spread(
    make_model, train_dl, test_dl, repeats: int = REPEATS, **kwargs
) -> Repeats:
    """Every metric of `repeats` runs, each from other random weights (nothing is
    seeded), and the mean seconds spent in training steps. `.values("accuracy")` takes
    one metric of every repeat."""
    return repeat(
        make_model, lambda model: run(model, train_dl, test_dl, **kwargs), repeats
    )


def compare(runs: dict[str, np.ndarray], title: str = "") -> None:
    """Repeated runs side by side, as a picture and as a table.

    Every entry of `runs` holds the accuracies of one model's repeats. The picture shows
    its mean, its standard deviation and every repeat. The first entry is the reference:
    the table gives, for every other entry, the difference with it in points and in
    sigmas (the difference divided by its standard error)."""
    names = list(runs)
    rows = [
        Comparison.of(name, runs[name], versus=runs[names[0]] if i else None)
        for i, name in enumerate(names)
    ]
    result_bars(
        runs, title=title, zoom=True, highlight=names[-1] if len(names) > 1 else None
    )
    plt.show()
    table = pd.DataFrame(
        {
            "mean": [r.mean for r in rows],
            "std": [r.std for r in rows],
            "runs": [len(r.values) for r in rows],
            "difference (pt)": [r.points for r in rows],
            "in sigma": [r.sigmas for r in rows],
        },
        index=names,
    )
    display(
        table.style.format(
            {
                "mean": "{:.4f}",
                "std": "{:.4f}",
                "difference (pt)": "{:+.2f}",
                "in sigma": "{:+.1f}",
            },
            na_rep="",
        )
    )


DIM = {enc: shape_of(DATASET, enc)[0] for enc in ENCODERS}
N_CLASSES = shape_of(DATASET, "dinov2-small")[1]

In [ ]:
USED = {*HEAD, "dataset", "encoder", "epochs", "n_train"}


def objective(space: Space, log: StudyLog):
    """What an Optuna study calls once per trial: draw a config from `space`, train it,
    report the accuracy after every epoch, and return the last one. Every trial logs
    every metric, the accuracy is the one the search maximises."""
    unused = sorted(set(space) - USED)
    if unused:
        raise ValueError(
            f"objective does not use {unused}: build it in make_head (and give "
            "it a default in HEAD), or take it out of the space"
        )

    def run_trial(trial: optuna.Trial) -> float:
        config = space.draw(trial)
        tr, te = loaders(
            config["dataset"], config["encoder"], n_train=config.get("n_train")
        )
        dim, n_classes = shape_of(config["dataset"], config["encoder"])
        model = make_head(config, dim, n_classes)
        seen = None

        # report every epoch, not only the last: the trial's learning curve reaches the
        # store, and the study's pruner can stop a trial that is behind the others
        def to_optuna(epoch, fit):
            nonlocal seen
            seen = fit
            trial.report(fit.scores().accuracy, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        try:
            training = Training(
                lr=config["lr"], epochs=config["epochs"], logdir=STORE.logdir()
            )
            seen = training.fit(model, tr, te, classes_of(te), to_optuna)
            return seen.scores().accuracy
        finally:
            # a pruned trial is recorded as well, with the epochs it ran and every
            # metric of its last epoch
            log.record(
                trial,
                RunRecord(
                    role="trial",
                    dataset=config["dataset"],
                    config=config,
                    fixed=space.fixed(),
                    train_seconds=seen.seconds if seen else 0.0,
                    curves=seen.curves if seen else None,
                    scores=seen.scores() if seen and seen.confusion else None,
                    confusion=seen.confusion if seen else None,
                    extra={"embed_dim": dim},
                ),
                Architecture.of(model, torch.zeros(2, dim)),
            )

    return run_trial


def search(
    space: Space,
    name: str,
    sampler=None,
    pruner=None,
    attrs=None,
    idea=None,
    n_trials: int = N_TRIALS,
) -> tuple:
    """Run `n_trials` trials of `space` as the Optuna study `name`, and return the study
    with a table of its finished trials (one row per trial). The study carries `idea`.

    `sampler=None` samples every trial at random and independently; pass
    `optuna.samplers.TPESampler()` for TPE. `pruner=None` lets every trial run all its
    epochs. Nothing is seeded: the trials drawn and the weights they start from differ
    per run of the notebook, so two students' searches of one space are two draws from
    it. A study that exists already gets the new trials added to it."""
    study = StudyLog.search(
        name=name, space=space, idea=idea, sampler=sampler, pruner=pruner, attrs=attrs
    )
    start = time.time()
    study.optimize(objective(space, study), n_trials=n_trials)

    done = study.trials(optuna.trial.TrialState.COMPLETE)
    pruned = study.trials(optuna.trial.TrialState.PRUNED)
    table = pd.DataFrame([{"Accuracy": t.value, **t.params} for t in done])
    print(
        f"{n_trials} trials in {time.time() - start:.0f}s; the study {name!r} now holds "
        f"{len(done)} finished and {len(pruned)} pruned"
    )
    return study, table

# Part 1 — the notebook as handed out

### First, look at the data you are about to slice

That `.shuffle(seed=0)` above is not defensive habit, it is load-bearing, and the reason is
worth two minutes because it is the kind of bug that produces a *plausible* wrong answer
rather than a crash.

In [ ]:
dtd = load(DATASET, "dinov2-small", "train")
labels = dtd["label"][:]
shuffled_labels = dtd.dataset.shuffle(seed=0)["label"][:]

print("first 20 labels in the DTD cache:", labels[:20].tolist())
print()
print(f"{'rows taken':>12} {'classes, as stored':>20} {'classes, shuffled':>19}")
for n in (470, 940, 1880):
    print(
        f"{n:>12} {len(set(labels[:n].tolist())):>14}/{N_CLASSES:<5} "
        f"{len(set(shuffled_labels[:n].tolist())):>13}/{N_CLASSES}"
    )

**The DTD cache is stored grouped by label.** Take the first 940 rows and you do not get a
smaller sample of the task — you get a *different, easier* task with a quarter of the
classes, and a test set full of categories your model has never once seen.

The reason this is dangerous rather than merely wrong is that it fails *quietly and
plausibly*. Subsampling this way at 470 / 940 / 1880 rows produces test accuracies of
0.12 / 0.24 / 0.44, which looks exactly like a healthy learning curve — right up until you
notice those numbers are just `n / 3760`. The model was scoring near-perfectly on the
classes it had and zero on the rest. Shuffled, the same sizes give about 0.69 / 0.75 / 0.78
(measured with a one-hidden-layer head), which is the real curve and a far less exciting one.

Two general lessons, both cheap:

- **Not every cache is shuffled.** The EuroSAT caches are (they were subsampled at build
  time, which shuffles); the DTD ones were built `--full`, which does not. Do not assume —
  it is one line to check, as above.
- **A number that looks like a textbook result deserves more suspicion, not less.** The
  broken version of this exercise produced a *nicer* curve than the correct one.

## E1 — The baseline, first

**Before you search anything**, establish the two-line baseline on DTD and its spread
between runs. A search without a baseline is unreadable; a baseline without a spread cannot tell
you whether a later "win" is real.

Every number you produce below gets compared against these two.

In [ ]:
train_dl, test_dl = loaders(DATASET, "dinov2-small")

baseline_runs = run_spread(
    lambda: nn.Linear(DIM["dinov2-small"], N_CLASSES), train_dl, test_dl
)
baseline = baseline_runs.values("accuracy")  # one accuracy per repeat

# stores the baseline: one row in the study lesson1-notebook-runs
example = next(iter(train_dl))[0][:2]  # two rows are enough to trace the shapes
RUNS.log(
    name="baseline",
    role="baseline",
    results=baseline_runs,
    dataset=DATASET,
    params=run_params(DATASET, "dinov2-small"),
    architecture=Architecture.of(nn.Linear(DIM["dinov2-small"], N_CLASSES), example),
)

compare(
    {f"baseline: Linear(384, 47), {REPEATS} runs": baseline},
    title=f"a later result must clear {baseline.mean() + 2 * baseline.std():.3f} "
    "(the mean + 2 std) before you should believe it beat this",
)

Both numbers, the mean and the spread, are now in the Optuna store (`RUNS.log` wrote
them), before anything else ran. That order is the point:
a baseline fixed in advance is the only thing that stops you from quietly redefining what
counted as success once you have seen the results.

## E2 — Argue from the metadata, then check

Notebook 01 said the metadata is the public API of a cache. This is not a cold guess —
you already have evidence, so use it.

**Before running the next cell**, answer for yourself:

1. Look back at notebook 01 §1.5's table: DINOv2 beat resnet-18 and mobilenet by 9-15
   points on *flowers*. Textures are a different domain, but the *ranking* of general-purpose
   image encoders tends to be fairly stable across natural-image tasks. Would you expect the
   same ordering here, or is there a reason textures specifically might reshuffle it? Either
   answer is defensible — the point is to have reasoned about it, not to have guessed right.
2. `mobilenet_v2` produces 1280 numbers per image against DINOv2's 384. Notebook 01 §1.5
   already told you width is not quality on flowers (`mobilenet_v2` lost by eleven points
   there despite more dimensions). Do you expect that to hold here too?
3. On flowers the encoder spread was about 15 points and on EuroSAT about 3. DTD's spread is
   somewhere in that range — pick a number and say why textures should be closer to one end
   than the other.

Then run it.

In [ ]:
results, n_weights = {}, {}
for enc in ENCODERS:
    tr, te = loaders(DATASET, enc)
    model = nn.Linear(DIM[enc], N_CLASSES)
    fit = run(model, tr, te)
    results[enc] = fit.scores().accuracy
    n_weights[enc] = sum(p.numel() for p in model.parameters())
    RUNS.log(
        name=f"encoder-{enc}",
        role="ablation",
        results=fit.as_repeats(),
        dataset=DATASET,
        params=run_params(DATASET, enc),
        architecture=Architecture.of(
            nn.Linear(DIM[enc], N_CLASSES), torch.zeros(2, DIM[enc])
        ),
    )

spread = max(results.values()) - min(results.values())
fig, axes = plt.subplots(1, 3, figsize=(14, 2.6))
result_bars(DIM, xlabel="width of the vector", fmt="{:,.0f}", ax=axes[0])
result_bars(n_weights, xlabel="weights of the linear head", fmt="{:,.0f}", ax=axes[1])
result_bars(results, xlabel="test accuracy, one run", ax=axes[2])
for ax in axes[1:]:
    ax.set_yticklabels([])
fig.suptitle(
    f"spread across the encoders: {100 * spread:.1f} pt; "
    f"the baseline's spread between runs: {100 * baseline.std():.2f} pt",
    x=0.01,
    ha="left",
    fontsize=10,
)
fig.tight_layout()
plt.show()

Compare the two numbers in the title. **How many times the spread between runs is the
encoder axis wide?** That ratio is the honest way to say "this axis matters and that one
does not". And look at the first two panels next to the third: is the widest vector, with
the most weights, the best one?

## E3 — The handed-out search

The search below is the one handed out to the whole class, on `dinov2-small` only. Every
axis is one of the ideas of this lesson; the links go to the picture of that learning goal.

| axis | values | the idea it tests | goal |
|---|---|---|---|
| `n_hidden_layers` | 0 to 3 | depth; 0 is the linear baseline | [1.6](../../presentations/leerdoelen/les1.html#g1-6) |
| `hidden_dim` | 32 to 1024 | width: more units, more capacity | [1.6](../../presentations/leerdoelen/les1.html#g1-6) |
| `activation` | ReLU, GELU, tanh, sigmoid | the nonlinearity between the layers | [1.6](../../presentations/leerdoelen/les1.html#g1-6) |
| `norm` | none, batch, layer | normalisation in the hidden block | |
| `skip` | off, on | a skip connection around the hidden block | [2.1](../../presentations/leerdoelen/les2.html#g2-1), lesson 2 |
| `dropout` | 0 to 0.5 | regularisation: switch off random units while training | [1.9](../../presentations/leerdoelen/les1.html#g1-9) |
| `sigma` | 0 to 0.5 | regularisation: noise on the input vectors | [1.9](../../presentations/leerdoelen/les1.html#g1-9) |
| `lr` | 1e-4 to 1e-2 | the size of the step the optimizer takes | [1.7](../../presentations/leerdoelen/les1.html#g1-7) |

Every trial trains for `EPOCHS` = 15 epochs.

In [ ]:
# The search space handed out to the whole class, searched at random. Part 2 starts from
# a copy of it. Both ends of a range are included.
HANDED_OUT_SPACE = Space(
    {
        "dataset": DATASET,
        "encoder": Choice(["dinov2-small"]),
        "n_hidden_layers": Int(0, 3),
        "hidden_dim": Int(32, 1024, log=True),  # built only when n_hidden_layers > 0
        "activation": Choice(["relu", "gelu", "tanh", "sigmoid"]),
        "norm": Choice(["none", "batch", "layer"]),
        "skip": Choice([False, True]),
        "dropout": Float(0.0, 0.5),
        "sigma": Float(0.0, 0.5),
        "lr": Float(1e-4, 1e-2, log=True),
        "epochs": EPOCHS,
    }
)

**The next lesson discusses these axes on the class's runs.** Everyone's trials are put
side by side, and we go through them together, axis by axis. Before then, make sure you
understand what your own search shows: for each axis, what did it do to accuracy and to
`train_seconds`, and does that match what the goal says it should? Where it does not, that
is the most interesting thing to bring.

Three things about how this search runs:

- **It samples at random, and it is not seeded.** Every trial is an independent draw from
  the space, so your 24 trials are other draws than your neighbour's. One search is one
  draw; fifteen show which choices win every time and which only won once. And because no
  trial depends on the ones before it, the class's trials pooled are one random search of
  a few hundred trials, dense enough to see a trend that 24 points cannot show.
- **Not TPE, as in notebook 04.** TPE samples where earlier trials scored well. That spends
  a small budget better, but fifteen TPE searches each crowd around their own first lucky
  draws, and pooled they are not a sample of the space.
- **No pruner.** A pruner stops a trial that is behind the others after a few epochs. A
  pruned trial has no final accuracy, and the class's pooled sample needs every draw to
  count, the bad ones too.

In [ ]:
# runs the search; every trial is stored in the study lesson1-notebook
handed_out, df = search(
    space=HANDED_OUT_SPACE,
    name="lesson1-notebook",
    attrs={"origin": "notebook"},
    idea=HANDED_OUT,
)

display(
    df.sort_values("Accuracy", ascending=False)
    .head(6)
    .style.format(
        {"Accuracy": "{:.3f}", "dropout": "{:.2f}", "sigma": "{:.2f}", "lr": "{:.1e}"}
    )
    .hide(axis="index")
    .set_caption("the six best trials")
)
compare(
    {
        f"baseline, {REPEATS} runs": baseline,
        "best trial of the search, one run": np.array([df["Accuracy"].max()]),
    },
    title="the search against the baseline",
)

In [ ]:
# Same plot as notebook 04 §4.2, generalised to whatever columns a search space produces --
# one line per trial, every axis at once, so an interaction between two axes is still
# visible even though no single-axis panel would show it.
fig = search_parallel_plot(df, columns=list(df.columns), color="Accuracy")
fig.update_layout(title="The handed-out search, every trial and every axis at once")
fig.show()

### Had the models finished learning?

Every trial trained for `EPOCHS` = 15 epochs. Whether that was enough is something you
read from the **learning curves**: the loss on the training set and on the test set after
every epoch ([goal 1.9](../../presentations/leerdoelen/les1.html#g1-9)).

- A train loss that is **still falling** at the last epoch: the model had not finished,
  and more epochs could have given a better accuracy.
- A train loss that is **flat**: it had finished learning what it could.
- A test loss that **turns up** while the train loss keeps falling: the model is
  overfitting, it is memorising the training set.

Below: the four best trials in the top row and the four worst in the bottom row. Optuna
has the same curves for every trial, under **History → Intermediate values**.

In [ ]:
finished = sorted(
    handed_out.trials(optuna.trial.TrialState.COMPLETE), key=lambda t: t.value
)
shown = [*finished[::-1][:4], *finished[:4]] if len(finished) >= 8 else finished[::-1]


def label(trial) -> str:
    p = trial.params
    return (
        f"trial {trial.number}: accuracy {trial.value:.3f}\n"
        f"{p['n_hidden_layers']} layers of {p['hidden_dim']}, {p['activation']}, "
        f"lr {p['lr']:.0e}"
    )


plot_learning_curves(
    {
        label(t): {k: t.user_attrs[k] for k in ("train_loss", "test_loss")}
        for t in shown
    },
    title="top row: the best trials; bottom row: the worst",
    sharey=False,
)
plt.show()

**Did anything beat the baseline? By how much?**

Careful with the comparison above. The best trial is a **single run**, selected
as the maximum of 24 draws. The baseline is a **mean over several runs**. Taking the max
of many noisy numbers and comparing it to an average is one of the most common ways to
manufacture a win that is not there — the max of 24 draws from the baseline's own
distribution would already sit above the baseline's mean.

Which is exactly what E4 is for.

## E4 — Rerun the winner honestly

Take the winning configuration and rerun it a few times. Then compare like with like.

In [ ]:
best = handed_out.best()
display(
    pd.DataFrame([{k: v for k, v in best.items() if k != "epochs"}])
    .style.hide(axis="index")
    .set_caption("the winning config")
)
best_dim, _ = shape_of(best["dataset"], best["encoder"])


def make_winner():
    return make_head(best, best_dim, N_CLASSES)


win_tr, win_te = loaders(best["dataset"], best["encoder"])
winner_runs = run_spread(make_winner, win_tr, win_te, lr=best["lr"])
winner = winner_runs.values("accuracy")

compare(
    {
        f"baseline, {REPEATS} runs": baseline,
        "the winner's single run in the search": np.array([df["Accuracy"].max()]),
        f"the winner rerun, {REPEATS} runs": winner,
    },
    title="the winner: its one run in the search, and the same config rerun",
)

In [ ]:
# The number that stands for this search: the winner rerun a few times, not its single-run
# leaderboard score. It sits in `lesson1-notebook-runs`, next to the baseline.
RUNS.log(
    name="winner",
    role="final",
    results=winner_runs,
    dataset=best["dataset"],
    params=run_params(best["dataset"], best["encoder"], best),
    architecture=Architecture.of(make_winner(), torch.zeros(2, best_dim)),
    attrs={"search_space": HANDED_OUT_SPACE.line()},
)

**This is the number that stands for the search**, not the leaderboard number.

Answer these for yourself before moving on:

1. How much of the apparent win survived being rerun? If it shrank, by how much?
2. Is the difference more than about two sigma (the last column of the table)? If not, what is the honest
   one-sentence conclusion?
3. Most "wins" at this scale are inside noise between runs. Is this one? Commit to an answer.

## E5 — An ablation you can actually settle

An **ablation** takes one part out of a model and measures what changes: the difference
is what that part was worth. Here the part is the noise on the input vectors.

`GaussianNoise` is the one head-side knob with any prior evidence behind it: earlier
measurements had it worth about +0.4 points on DTD. That is small enough that it is a
genuinely open question rather than a known result.

**Predict first:** does input noise still help once the training set is full size? Argue
from what noise is *for* — then test it at two training sizes.

In [ ]:
FULL = 3760


def with_noise(sigma):
    return lambda: Serial(
        [GaussianNoise(sigma=sigma), nn.Linear(DIM["dinov2-small"], N_CLASSES)]
    )


def train_accuracy(model, train_dl) -> float:
    return Scorer(train_dl, classes_of(train_dl))(model).scores().accuracy


noise_curves = {}
for n_train in (param(940, test=200), None):
    tr, te = loaders(DATASET, "dinov2-small", n_train=n_train)
    rows = n_train or FULL
    # scale epochs so every training size gets a comparable number of gradient steps,
    # otherwise you are measuring "less data" and "less training" at the same time
    epochs = max(EPOCHS, round(EPOCHS * FULL / rows))

    accuracies = {}
    for sigma in (0.0, 0.3):
        runs = run_spread(with_noise(sigma), tr, te, epochs=epochs)
        accuracies[f"sigma = {sigma}"] = runs.values("accuracy")
        RUNS.log(
            name=f"noise-{sigma}-{rows}rows",
            role="ablation",
            results=runs,
            dataset=DATASET,
            params=run_params(
                DATASET, "dinov2-small", {"sigma": sigma}, n_train=rows, epochs=epochs
            ),
            architecture=Architecture.of(
                with_noise(sigma)(), torch.zeros(2, DIM["dinov2-small"])
            ),
        )
        # one more run of this setting, for its learning curve and for its accuracy
        # on the training set
        fitted = with_noise(sigma)()
        fit = run(fitted, tr, te, epochs=epochs)
        noise_curves[
            f"{rows} rows, sigma = {sigma}\n"
            f"accuracy: train {train_accuracy(fitted, tr):.3f}, "
            f"test {fit.scores().accuracy:.3f}"
        ] = fit.curves.model_dump()

    compare(accuracies, title=f"{rows} training rows, {epochs} epochs")

plot_learning_curves(
    noise_curves, title="train and test loss per epoch, without and with noise"
)
plt.show()

Note the epoch scaling in that cell. Without it you would be changing two things at once —
less data *and* fewer gradient steps — and could not attribute the difference to either.
Was that the right call?

Three things to take from the output, and the third is the one worth keeping.

**Check whether either difference cleared its sigma.** If it did not, the honest sentence is
"no measurable effect at this training size", not "noise helped slightly". Resist the pull
of the sign: a +0.1 pt difference at 0.4 sigma is not a small benefit, it is no evidence.

**"Does GaussianNoise help?" is not a question with an answer.** "Does it help at *this*
training size, on *this* representation?" is. Regularisation is supposed to buy something
exactly when a model can memorise its training set — so look at the learning curves, and
at the training accuracy in their titles.

**The head reaches ~100% training accuracy and the test number does not move.** That is the
informative result. The model is memorising completely, and regularising it harder changes
nothing downstream, which means the ceiling you are hitting is not head capacity — it is
what the frozen representation makes linearly separable. You cannot regularise your way to
a better representation. Notebook 04 reached the same conclusion from the other direction:
the encoder moves accuracy by more than ten points, the head knobs by about two.

That is the end of the handed-out part. Everything it stored is in two Optuna
studies: `lesson1-notebook`, the trials of the search, and `lesson1-notebook-runs`, the
baseline, the encoder comparison, the winner and the noise ablation.

# Part 2 — your own study

**What is asked of you:** one study of your own. One idea, a design that can test it, a
search, and the result against a baseline, handed in with the runs of Part 1.

You do not do this in notebook cells. Your study lives in the folder
[`exercises/`](exercises/), next to this notebook, in two files you edit:

| file | what it is | what you do with it |
|---|---|---|
| [`exercises/designs.py`](exercises/designs.py) | the head, written as one typed method per stage, the way `fusion_architecture.py` is in notebook 04 §4.4. A block the search may leave out is marked `@option`. | read it with atlas; change it, or add a stage, when your idea needs another block |
| [`exercises/study.py`](exercises/study.py) | six settings at the top (`STUDY`, `IDEA`, `EXPECTS`, `OBJECTIVE`, `SPACE`, `MINUTES`), and below them the code that runs the study | fill in the six settings, then run it with `make study` |
| [`exercises/harness.py`](exercises/harness.py) | the caches, the trainer and the linear baseline | nothing: it is given |

## Where you start: what we have seen so far

| what we saw | where |
|---|---|
| The encoder matters most: on DTD the DINOv2 encoders beat resnet-18 by more than ten points. | 04 §4.2, E2 |
| One hidden layer on `dinov2-small` is worth about two points over the linear head. | 04 §4.2, your E4 |
| Fusing `dinov2-small` with resnet-18 is worth about two points. | 04 §4.3 |
| `sigma` and `lr` did not organise the trials inside their ranges; noise did not help a linear head. | 04 §4.2, E5 |
| Your own search: which axes were flat, and which moved the accuracy? | E3 |

All of that was measured on one dataset, DTD. Your study takes **one** of these findings
and asks: **does it still hold when I change one thing?** For example:

- another dataset: `eurosat`, `oxford-flowers` or `fashion-mnist`. Does a hidden layer
  still help on EuroSAT, where the linear head already scores about 0.94?
- less training data (`n_train`): does the gap between two options grow or shrink?
- another block in the head: a skip connection, a gate, a normalisation layer.

One question is one study. A second question is a second study.

## The loop

Your study is one round of this loop. Blue is your design, orange your study, green the
class. Every box links to its step in [your study, step by step](https://raoulg.github.io/MADS-MachineLearning-course/lesson1-your-study.html).

In [ ]:
study_loop("lesson1-your-study.html")

## The steps, one by one

Run every command in a terminal, from the root of the repository.

**1. Design: read the head with atlas.** Open the design as a picture before you change
or train anything:

```bash
atlas-code-nightly view notebooks/1_representation_and_head/exercises
```

Click `Head.forward`. You see the stages (`perturb`, `hidden`, `skip`, `gate`, `classify`)
and the tensor shape between them, and where a choice leaves a stage out.
([goal 1.12](../../presentations/leerdoelen/les1.html#g1-12))

**2. Build: change the design, if your idea needs it.** A study about another dataset or
about depth needs no change: go to step 3. A study about a block the head does not have
yet needs a new stage in `designs.py`; the comment at the top of that file says how, in
three steps. Then check that every combination builds, and read it in atlas again:

```bash
uv run notebooks/1_representation_and_head/exercises/designs.py
```

**3. Baseline.** Nothing to do: `study.py` measures the linear head on your dataset,
`REPEATS` times, before it searches. ([goal 1.13](../../presentations/leerdoelen/les1.html#g1-13))

**4. Fill in the six settings** at the top of `exercises/study.py`:

- `STUDY`: the name of your study, your first name, a dash, and what it studies:
  `"sam-hidden-layer-on-eurosat"`. It is the name you find it under in the dashboard.
- `IDEA`: what you investigate, in one sentence of your own. From your GitHub login and
  that sentence, `hpo` makes a short name for the idea, such as `blissful-bee-18`. A
  classmate who wants to continue where you stopped names that idea, and so the class
  can see who built on whom. Your first idea builds on Part 1, and that is filled in
  already (`builds_on=Idea.handed_out(1).name`).
- `EXPECTS`: what you expect to happen and why, in one line, written before you run. It
  is stored with the study and never graded; it is there so the result can surprise you.
- `OBJECTIVE`: the metric the search maximises. Leave it at `"accuracy"`.
- `SPACE`: your search space: which axes you search, over which ranges, and which you
  fix to one value. ([goal 1.14](../../presentations/leerdoelen/les1.html#g1-14))
- `MINUTES`: the time you give the study on your machine. The plot below shows what
  your minutes buy.

`study.py` stops with a message as long as `STUDY`, `IDEA` or `EXPECTS` still hold their
placeholder. This is a filled-in example that runs; start from it and make it yours:

```python
STUDY = "sam-hidden-layer-on-eurosat"
IDEA = Idea.of(
    "does a hidden layer still help on eurosat", builds_on=Idea.handed_out(1).name
)
EXPECTS = "less than on DTD: the linear head is already at 0.94, so little is left to gain"
OBJECTIVE = "accuracy"
SPACE = Space(
    {
        "dataset": "eurosat",
        "encoder": Choice(["dinov2-small"]),
        "n_hidden_layers": Int(1, 2),
        "hidden_dim": Int(32, 1024, log=True),
        "activation": "gelu",
        "norm": "none",
        "dropout": 0.0,
        "sigma": 0.0,
        "skip": False,
        "gate": False,
        "lr": Float(1e-4, 1e-2, log=True),
        "epochs": EPOCHS,
    }
)
MINUTES = 15
```

This example searches three axes (`n_hidden_layers`, `hidden_dim`, `lr`) and fixes the
rest. That is a choice: every axis you search needs trials, and your minutes buy only so
many. Set `MINUTES` and `AXES` in the cell below to see how many.

In [ ]:
MINUTES = 15  # the time you give your study on this machine
AXES = 3  # how many axes your SPACE searches

# what one trial costs on this machine, as measured by the search of Part 1
PART1 = StudyLog.load(name="lesson1-notebook")
SECONDS_PER_TRIAL = PART1.seconds_per_trial(epochs=EPOCHS)


def trials_in(minutes: float) -> int:
    """The trials that fit in `minutes`, once the baseline and the winner's rerun
    (REPEATS runs each) are set aside."""
    return Budget.at(
        minutes=minutes,
        seconds_per_trial=SECONDS_PER_TRIAL,
        measured_on=PART1.name,
        epochs=EPOCHS,
        repeats=REPEATS,
    ).n_trials


minutes = np.arange(1, 61)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(minutes, [trials_in(m) for m in minutes], color="tab:blue")
ax.scatter([MINUTES], [trials_in(MINUTES)], color="tab:orange", s=80, zorder=3)
ax.annotate(
    f"{MINUTES} minutes: {trials_in(MINUTES)} trials, "
    f"{trials_in(MINUTES) / AXES:.0f} per searched axis",
    (MINUTES, trials_in(MINUTES)),
    xytext=(10, -14),
    textcoords="offset points",
)
ax.set_xlabel("minutes you give the study")
ax.set_ylabel("trials that fit")
per_axis = ax.secondary_yaxis(
    "right", functions=(lambda t: t / AXES, lambda a: a * AXES)
)
per_axis.set_ylabel(f"trials per axis, with {AXES} searched axes")
ax.spines[["top"]].set_visible(False)
ax.set_title(
    f"one trial costs {SECONDS_PER_TRIAL:.1f}s here, at {EPOCHS} epochs "
    "(measured in Part 1, on DTD with dinov2-small)",
    loc="left",
    fontsize=10,
)
fig.tight_layout()
plt.show()

A few trials per axis cannot tell a good range from a lucky draw. If the number on the
right is small, fix an axis to one value or give the study more minutes. The class page
(`make dashboard LESSON=1`) pools everyone's handed-out search: an axis that is flat there
is the first one to fix. A larger dataset or encoder costs more per trial than the plot
says.

**5. Search.**

```bash
make study
```

This runs `exercises/study.py`: it checks your space against your design, measures the
baseline, runs the search, reruns the winner `REPEATS` times, and prints the winner
against the baseline in sigmas. Everything is stored, in the same file as Part 1, under
two studies: `lesson1-study-<STUDY>` (the trials) and `lesson1-study-<STUDY>-runs` (the
baseline and the winner). Every run carries your `designs.py`, so the class sees the
architecture next to the numbers.

**6. Read.** `make optuna-dashboard`, and open your study; the section below says what is
on each page. Read the winner in sigmas, not in points: inside about two, the difference
is not a result, whatever its sign. ([goal 1.10](../../presentations/leerdoelen/les1.html#g1-10))

**7. Share.** `make share LESSON=1` hands in Part 1 and your study together; see the last
section. ([goal 1.15](../../presentations/leerdoelen/les1.html#g1-15))

**8. Build on.** `make dashboard LESSON=1` opens the class page. An idea of a classmate
that you want to continue becomes the `builds_on` of your next `IDEA`, and that next idea
is a new `STUDY`.

## Working with Claude, or another coding agent

You may do Part 2 together with a coding agent. The repository tells it how to help you
(`CLAUDE.md` and `AGENTS.md`, which it reads by itself): it coaches, and you decide. Before
it writes a search space or a stage it asks you three things: what do you investigate,
what do you expect and why, and which idea of the class do you build on? Your answers
become `IDEA`, `EXPECTS` and `builds_on`.

A good way to start: open the repository in your agent and say *"I want to do Part 2 of
lesson 1's exercise; start at https://raoulg.github.io/MADS-MachineLearning-course/lesson1-your-study.html"*.

The rule that decides whether you used it well: **you can explain every stage of your
design and every axis of your space without the agent present.** In the next lesson you
are asked three things about your study:

1. **Why did you do this?** What made this the question worth your budget?
2. **What did you expect, and from what?**
3. **What did you learn, also when it disappointed?** A search that found nothing, or a
   winner that fell back into the baseline's spread once rerun, says something about the
   representation, the space or the idea.

---

## Read your search in optuna-dashboard

Every cell above wrote to `artefacts/optuna/studies.db`. From the root of the course
repository:

```bash
make optuna-dashboard
```

and open http://127.0.0.1:8088. Four studies are there:

| study | what it holds |
|---|---|
| `lesson1-notebook` | the trials of the handed-out search |
| `lesson1-notebook-runs` | the baseline, the encoder comparison, the winner rerun a few times, the noise ablation |
| `lesson1-study-<STUDY>` | the trials of your own search, the pruned ones included |
| `lesson1-study-<STUDY>-runs` | your baseline and your winner |

Open a study; its pages are in the bar on the left:

- **History**: the accuracy of every trial in the order they ran with the best so far,
  which axis explains most of the differences in accuracy (**Hyperparameter
  Importance**; with a few dozen trials a hint, to check against the baseline's spread between runs), and
  **Intermediate values**: every trial's accuracy per epoch, one line per trial. In your
  own study the pruned trials stop early; in a `-runs` study the steps are the repeats of
  one run, so the height of a line's wiggle is its spread between runs.
- **Analytics**: accuracy against one param (Slice) and against two (Contour, Rank).
- **Smart Selection**: the parallel-coordinates plot. The params are on the axes, and the
  numeric user attrs `n_params` and `train_seconds` can be ticked as axes too. Drag along
  an axis to keep the trials in that range; the history plot and the table below follow.
- **Trials (List)**: one page per trial, with a note that holds the layer table and the
  tensor shape at every layer, the accuracy per epoch, the params, the user attrs (`role`,
  `dataset`, `n_params`, `architecture`, `train_seconds`, the loss curves) and the
  artifact `architecture.json`.

The study's own page shows its user attrs: its idea (`idea`, the words, and `idea_name`,
the name a classmate builds on, next to `idea_author`, `idea_parents` and `idea_id`), and
for a search `search_space` and `searched_axes`. Your own study adds `expects`, what
you wrote in `EXPECTS`, and `budget`: the minutes you gave it, the seconds per trial
measured in Part 1, and `n_trials`, the trials it ran.

## Hand in your runs before the next lesson

Your runs are still only on your machine, in `artefacts/optuna/studies.db`: Part 1 in
`lesson1-notebook` and `lesson1-notebook-runs`, Part 2 in `lesson1-study-<STUDY>` and
`lesson1-study-<STUDY>-runs`. Handing in sends those four studies to the class runs
repository on GitHub.

The first time, this needs a one-time setup: a GitHub account, the `gh` command logged
in, and your own copy (a fork) of the class runs repository. `docs/submit-runs.md` has
the steps, and a coding agent can do them with you: *"follow docs/submit-runs.md for
lesson 1"*.

From the root of the course repository, first look at what would be handed in. This
changes nothing:

```bash
make share LESSON=1 ARGS=--list
```

Then hand them in:

```bash
make share LESSON=1
```

What that command does: it runs `scripts/submit_runs.py`, which reads the trials of the
four studies, writes each as a small JSON file (`lesson1/<your login>/<run_id>.json`: the
params, the accuracy per epoch, the architecture, your design script), commits them to
your fork, and opens a pull request to the class repository. A check there validates the
files and merges them. The last line it prints is the address of that pull request.
Handing in again later is safe; only new runs are added.

The next lesson opens on the whole class's runs: everyone's handed-out runs side by
side (they should agree, up to the spread between runs and between machines), and every
study with its search space, its baseline and its final. Your runs are part of that
picture, including the ones that lost. To look at the class's runs yourself:
`make dashboard LESSON=1`.